# Hue, lightness, and saturation count extraction

In this script,
we compute a subsampled hue, lightness and saturation count time series for an input video file.
In non-technical terms,
every few frames,
we count the number of pixels on screen having a specific hue, a specific lightness, and a specific saturation, throughout the entire video.

We tally over the HLS color space (over say RGB) because it is conceptually and perceptually a more intuitive and reasonable color space that should offer more insight into the director's design choices:
it seems more reasonable that the director thinks hard about a scene having a specific saturation and lightness, rather than the scene having a specific number of pixels having a strong green component.

In [ ]:
# import libraries
import os
import pathlib

import numpy as np
import cv2 as cv
import tqdm

from video import Video

In [ ]:
# define parameters
# We sample every STRIDE=6 frames.

STRIDE = 6
NAME = "hls_counts"

In [ ]:
# specify file structure
base_dir = pathlib.Path(os.getcwd())
input_dir = base_dir / "input"
output_dir = base_dir / "output"

In [ ]:
# read video
video = Video(input_dir / "perfect_blue.mp4", verbose=True)

In [ ]:
# tally hue, lightness, and saturation counts for each frame
# This takes around twenty minutes.

out = np.zeros((video.num_frames // STRIDE, 3, 256), dtype=int)

for i in tqdm.trange(video.num_frames // STRIDE):
    frame = video[STRIDE * i]
    hls = cv.cvtColor(frame, cv.COLOR_BGR2HLS).reshape(-1, 3)

    np.add.at(out[i, 0], np.clip(hls[:, 0], 0, 179), 1)
    np.add.at(out[i, 1], np.clip(hls[:, 1], 0, 255), 1)
    np.add.at(out[i, 2], np.clip(hls[:, 2], 0, 255), 1)

In [ ]:
# save counts
np.save(output_dir / NAME, out)